<a href="https://colab.research.google.com/github/WahyuKhairi06/BigData_2311531009_Wahyu-Khairi/blob/main/TB%20Praktikum/Tugas_Akhir_Praktikum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ==============================
# 1. IMPORT LIBRARY
# ==============================

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# ==============================
# 2. LOAD DATASET
# ==============================

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/WahyuKhairi06/BigData_2311531009_Wahyu-Khairi/refs/heads/main/TB%20Praktikum/healthcare-dataset-stroke-data.csv")

# ==============================
# 3. DATA CLEANING
# ==============================

In [3]:
df = df.drop(columns=['id'])
df = df[df['gender'] != 'Other']

# ==============================
# 4. SPLIT FITUR & TARGET
# ==============================

In [4]:
X = df.drop(columns=['stroke'])
y = df['stroke']

# ==============================
# 5. DEFINISI KOLOM
# ==============================

In [5]:
numerical_features = ['age', 'avg_glucose_level', 'bmi']
categorical_features = [
    'gender',
    'ever_married',
    'work_type',
    'Residence_type',
    'smoking_status'
]

# ==============================
# 6. PIPELINE PREPROCESSING
# ==============================

In [6]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

# ==============================
# 7. MODEL DASAR
# ==============================

In [7]:
logreg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', logreg)
])

# ==============================
# 8. TRAIN TEST SPLIT
# ==============================

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# ==============================
# 9. GRIDSEARCHCV
# ==============================

In [10]:
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear']
}
grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [11]:
# ==============================
# 10. TRAIN DENGAN TUNING
# ==============================

In [12]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['age',
                                                                          'avg_glucose_level',
                                                                          'bmi']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['gender',
                                                                          'ever_married',
                                                                          'work_type',
                                                                          'Residence_type',
                                                                          'smoking_status'])])),
                                       ('classifier',
                                        LogisticRegression(class_weight='balanced',
                                                           max_iter=1000,
                                                           random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__C': [0.01, 0.1, 1, 10],
                         'classifier__penalty': ['l2'],
                         'classifier__solver': ['liblinear']},
             scoring='f1', verbose=1)

# ==============================
# 11. MODEL TERBAIK
# ==============================

In [13]:
best_model = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)


Best Parameters:
{'classifier__C': 1, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}


In [14]:
# ==============================
# 12. EVALUASI MODEL TERBAIK
# ==============================

In [17]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.73      0.84       972
           1       0.13      0.78      0.22        50

    accuracy                           0.73      1022
   macro avg       0.56      0.75      0.53      1022
weighted avg       0.94      0.73      0.81      1022

ROC-AUC: 0.8396296296296297


# ==============================
# 13. SIMPAN MODEL
# ==============================

In [18]:

joblib.dump(best_model, "stroke_model.joblib")
print("\nModel terbaik berhasil disimpan sebagai stroke_model.joblib")


Model terbaik berhasil disimpan sebagai stroke_model.joblib
